# Variable-Channel Spatial ASR — 학습 및 본 평가 통합 노트북

화자 구분 비교 사다리의 모든 시스템을 이 노트북 하나로 학습·검증·본 평가(`eval-t2`)까지 자동 실행합니다.

- **`CONFIG = "all"` (기본값)**: 비교군 3종(`voice_only` → `space_only` → `naive_fusion`)과 제안 모델 3종(`proposed_carry` → `proposed_reset` → `proposed_carry_nodrop`)을 **한 번의 실행으로 순서대로 전부** 돌립니다.
- **중간에 Colab이 끊겨도 안전합니다**: 모든 체크포인트, 완료된 `summary.json`, RIR 뱅크, 데이터 캐시, ASR 전사 캐시가 Google Drive(`/content/drive/MyDrive/vcsasr`)에 실시간 저장됩니다. 런타임이 초기화되면 **그냥 다시 `런타임 > 모두 실행`**만 누르세요. 이미 끝난 모델은 건너뛰고, 학습 중이던 모델은 마지막 체크포인트에서 자동으로 이어서 진행합니다.

| CONFIG | 시스템 | 학습 여부 |
|---|---|---|
| `all` | **전체 6개 본 실험 시스템 순차 자동 실행** (`voice_only` ~ `proposed_carry_nodrop`) | 자동 |
| `all_baselines` | 무학습 비교군 3종(`voice_only`, `space_only`, `naive_fusion`)만 순차 실행 | 임계값·가중치만 조정 |
| `all_proposed` | 제안 모델 3종(`proposed_carry`, `proposed_reset`, `proposed_carry_nodrop`)만 순차 실행 | 학습 |
| `smoke_real` | GPU 사전 점검: 실제 ECAPA · pyroomacoustics · 실제 크기 모델로 30걸음 (약 5~10분) | 학습 (점검용) |
| `voice_only` | 목소리 전용: 동결 ECAPA + 온라인 군집 | 임계값만 조정 |
| `space_only` | 공간 전용: SRP-PHAT 방위 분포 + 온라인 군집 | 임계값만 조정 |
| `naive_fusion` | 단순 결합: 두 유사도를 고정 가중치로 합산 | 가중치·임계값만 조정 |
| `proposed_carry` | 제안 모델: TAC + ECAPA + GRU, 청크 간 상태 유지 (Carry-Over) | 학습 |
| `proposed_reset` | 제안 모델: 같은 구조, 청크마다 상태 초기화 (Chunk-Reset) | 학습 |
| `proposed_carry_nodrop` | 제안 모델 대조군: 4채널 고정 학습 (채널 드롭아웃 없음) | 학습 |

저장소: https://github.com/hd1534/variable-channel-spatial-asr

## 0. 설정

그대로 두고 `런타임 > 모두 실행`을 누르면 전체 실험(`all`)과 본 평가셋 채점(`RUN_EVAL_T2 = True`)이 한 번에 돌아갑니다.

In [ ]:
CONFIG = "all"  # @param ["all", "all_baselines", "all_proposed", "smoke_real", "smoke", "voice_only", "space_only", "naive_fusion", "proposed_carry", "proposed_reset", "proposed_carry_nodrop"]
RUN_EVAL_T2 = True  # @param {type:"boolean"}
# 학습 화자. train-clean-100(251명, 6.3 GB)이 기본입니다. dev-clean-only는 큰 내려받기 없이
# dev-clean의 평가 제외 화자 34명을 학습 24명 · 검증 10명으로 나눠 빨리 돌려 보는 용도입니다.
SPEAKERS = "train-clean-100"  # @param ["train-clean-100", "dev-clean-only"]
# 설정 덮어쓰기 (공백으로 구분). 예: train.max_steps=2000 train.batch_size=8
OVERRIDES = ""  # @param {type:"string"}
RUN_TAG = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
COMMIT = ""  # @param {type:"string"}
ALLOW_CODE_CHANGE = False  # @param {type:"boolean"}

REPO_URL = "https://github.com/hd1534/variable-channel-spatial-asr.git"
DRIVE_ROOT = "/content/drive/MyDrive/vcsasr"  # 체크포인트 · 데이터 캐시 (런타임이 끊겨도 남는다)
DATA_ROOT = "/content/data"                   # 학습 중 읽는 데이터 (로컬 디스크가 Drive보다 훨씬 빠르다)
REPO_DIR = "/content/variable-channel-spatial-asr"

PRESETS = {
    "all": ["voice_only", "space_only", "naive_fusion", "proposed_carry", "proposed_reset", "proposed_carry_nodrop"],
    "all_baselines": ["voice_only", "space_only", "naive_fusion"],
    "all_proposed": ["proposed_carry", "proposed_reset", "proposed_carry_nodrop"],
}
CONFIG_LIST = PRESETS.get(CONFIG, [CONFIG])
print("실행할 CONFIG 목록:", CONFIG_LIST, "| 본 평가(eval-t2) 실행:", RUN_EVAL_T2)

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path


def sh(cmd, check=True):
    """셸 명령을 실행하고 출력을 셀에 바로 보여 준다. 실패하면 셀을 멈춘다."""
    print("$", cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0 and check:
        raise RuntimeError(f"실패 (종료 코드 {proc.returncode}): {cmd}")
    return proc.returncode


IN_COLAB = "google.colab" in sys.modules
import torch

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"Colab: {IN_COLAB} | torch {torch.__version__} | GPU: {GPU or '없음'}"
      + (f" | compute capability {torch.cuda.get_device_capability()}" if GPU else ""))
if IN_COLAB and GPU is None and any(c.startswith("proposed") or c == "smoke_real" for c in CONFIG_LIST):
    raise RuntimeError("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 고른 뒤 다시 실행하세요.")

## 1. Google Drive 연결과 저장소 받기

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

if not Path(REPO_DIR, ".git").is_dir():
    sh(f'git clone -q --depth 1 --filter=blob:none --no-checkout -b {BRANCH} {REPO_URL} "{REPO_DIR}"')
    if RUN_EVAL_T2:
        sh(f"git -C \"{REPO_DIR}\" sparse-checkout set --no-cone '/*' '!/data/eval-dev/audio/'")
    else:
        sh(f"git -C \"{REPO_DIR}\" sparse-checkout set --no-cone '/*' '!/data/*/audio/'")
elif RUN_EVAL_T2 and not Path(REPO_DIR, "data/eval-t2/audio/t2_0000.wav").is_file():
    sh(f"git -C \"{REPO_DIR}\" sparse-checkout set --no-cone '/*' '!/data/eval-dev/audio/'")
sh(f'git -C "{REPO_DIR}" fetch -q --depth 1 origin {COMMIT or BRANCH} && git -C "{REPO_DIR}" checkout -q FETCH_HEAD')
os.chdir(REPO_DIR)
sh("git log -1 --format='%h %s'")

## 2. 패키지 설치

Colab에 미리 설치된 `torch`·`torchaudio`·`numpy`는 그대로 두고(제약 파일로 고정), 동결 화자 인코더(ECAPA-TDNN)를
불러올 `speechbrain`만 설치합니다. 방 음향을 계산하는 `pyroomacoustics`는 Drive에 방 음향 뱅크가 없을 때만 3절에서 설치합니다.

In [ ]:
sh("pip freeze | grep -iE '^(torch|torchaudio|numpy)==' > /content/constraints.txt")
sh('pip -q install -c /content/constraints.txt "speechbrain==1.1.2"')
sh(f'{sys.executable} -c "import torchaudio, soundfile; from speechbrain.inference.speaker import EncoderClassifier; '
   f'import torch; print(\'torch\', torch.__version__, \'| speechbrain 설치 확인\')"')

## 3. 데이터 및 방 음향 뱅크 준비

- **LibriSpeech**: 압축 파일을 Drive(`vcsasr/cache/`)에 한 번 받아 두고, 매 세션 로컬 디스크(`/content/data`)로 풉니다.
- **방 음향 뱅크**: `pyroomacoustics`로 방 800개(학습용)와 64개(검증용)를 계산해 Drive(`vcsasr/rirbanks/`)에 둡니다. 50개 방이 끝날 때마다 Drive에 바로 복사되므로 중간에 끊겨도 이어서 만듭니다.
- **ECAPA 및 ASR 체크포인트**: Drive(`vcsasr/models/ecapa/`)와 로컬 캐시에 준비합니다.

In [ ]:
LIBRISPEECH_MD5 = {
    "dev-clean": "42e2234ba48799c1f50f24a7926300a1",
    "train-clean-100": "2a93770f6d5c6c964bc36631d331a522",
}
MIRRORS = ["https://www.openslr.org/resources/12", "https://openslr.elda.org/resources/12",
           "https://openslr.trmal.net/resources/12"]

need = []
for c in CONFIG_LIST:
    cfg_json = json.loads(Path(f"configs/{c}.json").read_text(encoding="utf-8"))
    trainable = cfg_json.get("model", {}).get("name", "proposed") == "proposed"
    synthetic = bool(cfg_json.get("data", {}).get("synthetic"))
    own_train_root = "train_root" in cfg_json.get("data", {}) or synthetic
    if not synthetic and "dev-clean" not in need:
        need.append("dev-clean")
    if trainable and SPEAKERS == "train-clean-100" and not own_train_root and "train-clean-100" not in need:
        need.append("train-clean-100")

cache = Path(DRIVE_ROOT, "cache")
cache.mkdir(parents=True, exist_ok=True)
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
for name in need:
    if Path(DATA_ROOT, "LibriSpeech", name).is_dir():
        print("이미 풀려 있음:", name)
        continue
    cached = cache / f"{name}.tar.gz"
    if cached.is_file():
        tar = cached
    else:
        tar = Path(DATA_ROOT, f"{name}.tar.gz")
        for mirror in MIRRORS:
            if sh(f'curl -L --fail --retry 3 -C - -o "{tar}" {mirror}/{name}.tar.gz', check=False) == 0:
                break
        else:
            raise RuntimeError(f"{name}: OpenSLR 미러 세 곳 모두에서 받지 못했습니다.")
        digest = hashlib.md5()
        with open(tar, "rb") as fh:
            for block in iter(lambda: fh.read(1 << 22), b""):
                digest.update(block)
        if digest.hexdigest() != LIBRISPEECH_MD5[name]:
            tar.unlink()
            raise RuntimeError(f"{name}: md5가 OpenSLR 공개값과 다릅니다. 받은 파일을 지웠으니 이 셀을 다시 실행하세요.")
    staging = Path(DATA_ROOT, f".extract-{name}")
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    sh(f'tar -xzf "{tar}" -C "{staging}"')
    Path(DATA_ROOT, "LibriSpeech").mkdir(exist_ok=True)
    for item in Path(staging, "LibriSpeech").iterdir():
        target = Path(DATA_ROOT, "LibriSpeech", item.name)
        if not target.exists():
            item.rename(target)
    shutil.rmtree(staging, ignore_errors=True)
    if tar != cached:
        if shutil.disk_usage(DRIVE_ROOT).free > tar.stat().st_size + 2e9:
            shutil.copy2(tar, cached.with_name(cached.name + ".part"))
            os.replace(cached.with_name(cached.name + ".part"), cached)
        else:
            print(f"Drive 여유 공간이 부족해 {name} 압축 파일을 캐시하지 않습니다 (다음 세션에 다시 받습니다).")
        tar.unlink()

def build_run_info(cfg_name):
    cfg_json = json.loads(Path(f"configs/{cfg_name}.json").read_text(encoding="utf-8"))
    synthetic = bool(cfg_json.get("data", {}).get("synthetic"))
    own_train_root = "train_root" in cfg_json.get("data", {}) or synthetic
    sets = [f"--set {o}" for o in OVERRIDES.split()]
    dev_only = SPEAKERS == "dev-clean-only" and not own_train_root
    if dev_only:
        sets.append("--set data.train_root=LibriSpeech/dev-clean")
    run_name = cfg_name + ("-devclean" if dev_only else "") + (f"-{RUN_TAG}" if RUN_TAG else "")
    run_dir = f"{DRIVE_ROOT}/runs/{run_name}"
    args = (f'--config configs/{cfg_name}.json --data-root "{DATA_ROOT}" --run-dir "{run_dir}" '
            f'--model-cache "{DRIVE_ROOT}/models/ecapa" --bank-mirror "{DRIVE_ROOT}/rirbanks" '
            + ("--allow-code-change " if ALLOW_CODE_CHANGE else "") + " ".join(sets))
    return run_name, run_dir, args, dev_only

# 필요한 방 음향 뱅크 확인 및 생성
need_pra = False
for c in CONFIG_LIST:
    _, _, args, _ = build_run_info(c)
    if sh(f"{sys.executable} train.py {args} --check-banks", check=False) == 3:
        need_pra = True
if need_pra:
    sh('pip -q install -c /content/constraints.txt '
       '"pyroomacoustics @ git+https://github.com/LCAV/pyroomacoustics@v0.8.5"')

for c in CONFIG_LIST:
    _, _, args, _ = build_run_info(c)
    sh(f"{sys.executable} train.py {args} --prepare-only")

if RUN_EVAL_T2 and any(c not in ("smoke", "smoke_real") for c in CONFIG_LIST):
    sh("make asr-cache")

## 4. 순차 학습 · 검증 · 본 평가(`eval-t2`) 실행

`CONFIG_LIST`에 있는 시스템들을 순서대로 실행합니다.
- **이어하기(Resume) 자동 지원**: 이미 `summary.json`까지 완료된 시스템(학습 목표 스텝 도달)은 다시 학습하지 않고 즉시 넘어가며, 학습 도중 끊겼던 시스템은 `ckpt/LATEST.json`의 마지막 체크포인트에서 이어서 학습합니다.
- **`RUN_EVAL_T2 = True`일 때**: 각 시스템의 검증셋 임계값과 `best.pt`가 준비되는 즉시 본 평가셋(`eval-t2`, `eval-t2-static`) 추론·전사·채점을 실행하고 결과를 Drive(`vcsasr/eval/<RUN_NAME>/`)에 저장합니다. 이미 채점이 끝난 시스템은 건너뜁니다.

In [ ]:
for c in CONFIG_LIST:
    run_name, run_dir, args, dev_only = build_run_info(c)
    print("\n" + "=" * 78)
    print(f">>> [{run_name}] 학습 및 검증 시작")
    print("=" * 78, flush=True)

    summary_path = Path(run_dir, "summary.json")
    cfg_json = json.loads(Path(f"configs/{c}.json").read_text(encoding="utf-8"))
    max_steps = cfg_json.get("train", {}).get("max_steps", 20000)
    trainable = cfg_json.get("model", {}).get("name", "proposed") == "proposed"

    already_trained = False
    if summary_path.is_file():
        s_prev = json.loads(summary_path.read_text(encoding="utf-8"))
        if not trainable or s_prev.get("trained_steps", 0) >= max_steps:
            already_trained = True
            print(f"[skip] {run_name}: 이미 완료된 summary.json이 Drive에 있습니다 (steps={s_prev.get('trained_steps')}).")

    if not already_trained:
        sh(f"{sys.executable} train.py {args}")

    if RUN_EVAL_T2 and c not in ("smoke", "smoke_real"):
        eval_out = Path(DRIVE_ROOT, "eval", run_name)
        expected_variants = ["full", "learned_space", "fixed_gate"] if trainable else ["full"]
        eval_done = all(
            (eval_out / set_name / v / "score.json").is_file()
            for set_name in ("eval-t2", "eval-t2-static")
            for v in expected_variants
        )
        if eval_done:
            print(f"[skip] {run_name}: 본 평가(eval-t2, eval-t2-static) 채점 결과가 이미 Drive에 있습니다.")
        else:
            print(f">>> [{run_name}] 본 평가셋(eval-t2, eval-t2-static) 추론 및 채점 시작", flush=True)
            asr_cache_dir = f"{DRIVE_ROOT}/asr_cache"
            comp_flag = ""
            if c == "naive_fusion":
                suffix = ("-devclean" if dev_only else "") + (f"-{RUN_TAG}" if RUN_TAG else "")
                v_eval = Path(DRIVE_ROOT, "eval", "voice_only" + suffix)
                s_eval = Path(DRIVE_ROOT, "eval", "space_only" + suffix)
                if (v_eval / "eval-t2/embeddings.npz").is_file() and (s_eval / "eval-t2/embeddings.npz").is_file():
                    comp_flag = f'--components-from "{v_eval}" "{s_eval}"'
            sh(f'{sys.executable} eval_t2.py --run-dir "{run_dir}" --out "{eval_out}" '
               f'--device {"cuda" if GPU else "cpu"} --model-cache "{DRIVE_ROOT}/models/ecapa" '
               f'--asr-cache "{asr_cache_dir}" {comp_flag}')

# 마지막 실행된 CONFIG를 5절 상세 확인용 기본값으로 둔다
RUN_NAME, RUN_DIR, ARGS, DEV_CLEAN_ONLY = build_run_info(CONFIG_LIST[-1])

## 5. 개별 실행 상세 확인

마지막에 실행된 시스템(또는 `RUN_DIR`로 지정한 시스템)의 검증 상세 지표와 학습 곡선을 확인합니다.

In [ ]:
summary = json.loads(Path(RUN_DIR, "summary.json").read_text(encoding="utf-8"))
official = (summary.get("official_n2") or {}).get("speaker_attribution_accuracy")
print(f"{summary['run']}  ({summary['system']}, {summary['trained_steps']} steps, best at {summary.get('best_step')})")
print(f"Spk Acc(cap4) {summary['spk_acc']:.4f} | 2인 공식(cap2, evaluate.py) {official} | causal 상한 {summary['causal_ceiling_by_N']}")
print()
gate = summary.get("gate_by_M", {})
cap2_by_m = summary.get("spk_acc_n2_cap2_by_M", {})
theta_off = summary.get("theta_official_by_M", summary["theta_by_M"])
print(f"{'마이크 수':>8}{'2인 공식(cap2)':>14}{'내부(cap4)':>11}{'교대 직후':>10}{'턴 안':>8}{'공식 임계값':>10}" + (f"{'gate':>8}" if gate else ""))
for M in sorted(summary["spk_acc_by_M"]):
    row = (f"{M:>8}{cap2_by_m.get(M, summary['spk_acc_by_M'][M]):>14.4f}{summary['spk_acc_by_M'][M]:>11.4f}"
           f"{summary.get('spk_acc_after_change_by_M', {}).get(M, float('nan')):>10.4f}"
           f"{summary.get('spk_acc_steady_by_M', {}).get(M, float('nan')):>8.4f}{theta_off[M]:>10.2f}")
    print(row + (f"{gate.get(M, float('nan')):>8.3f}" if gate else ""))
diag = summary.get("diagnostics")
if diag:
    print()
    print("진단 (같은 검증 세션, 마이크 수별 Spk Acc)")
    for name, key in (("목소리 전용", "voice_only_by_M"), ("목소리 전용 대비 이득", "paired_gain_over_voice_by_M"),
                      ("학습된 공간 경로만", "learned_space_only_by_M"), ("고정 게이트", "fixed_gate_by_M")):
        print(f"  {name:<14}", diag[key])
    print("  정지·무잡음 조건 이득  ", diag["static"]["paired_gain_over_voice_by_M"])
    print("  한 위치 조건 이득      ", diag["colocated"]["paired_gain_over_voice_by_M"])
    print("  한 위치 조건, 공간 경로만의 화자 분리", diag["colocated"]["spatial_part_talker_acc_by_M"],
          f"(우연 수준 약 {diag['colocated']['chance_approx']})")

metrics = [json.loads(line) for line in open(Path(RUN_DIR, "metrics.jsonl"), encoding="utf-8")]
train_rows = [m for m in metrics if m.get("kind") == "train"]
val_rows = [m for m in metrics if m.get("kind") == "val"]
if train_rows:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for key in ("loss_spk", "loss_pos"):
        ax[0].plot([m["step"] for m in train_rows if key in m], [m[key] for m in train_rows if key in m], label=key)
    ax[0].set_title("train loss")
    ax[0].set_xlabel("step")
    ax[0].legend(fontsize=8)
    for M in "1234":
        pts = [(m["step"], m[f"gate_M{M}"]) for m in train_rows if f"gate_M{M}" in m]
        if pts:
            ax[1].plot(*zip(*pts), label=f"gate M={M}")
    if val_rows:
        ax[1].plot([m["step"] for m in val_rows], [m.get("spk_acc_n2_cap2", m["spk_acc"]) for m in val_rows], "k.-", label="val Spk Acc (cap2)")
    ax[1].set_xlabel("step")
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 6. 전체 시스템 비교표 (검증셋 및 본 평가셋 `eval-t2`)

Drive에 저장된 모든 시스템의 검증셋 성적과 본 평가셋(`eval-t2`, `eval-t2-static`) 성적을 한눈에 비교합니다.

In [ ]:
print("=== [검증셋 (dev-clean, 2인 공식 cap=2)] ===")
rows = []
for path in sorted(Path(DRIVE_ROOT, "runs").glob("*/summary.json")):
    s = json.loads(path.read_text(encoding="utf-8"))
    rows.append((path.parent.name, s))
print(f"{'run':<26}" + "".join(f"{'M=' + M:>9}" for M in "1234") + f"{'전체':>9}{'2인 공식':>10}")
for name, s in rows:
    by_m = s.get("spk_acc_n2_cap2_by_M") or s.get("spk_acc_by_M", {})
    official = (s.get("official_n2") or {}).get("speaker_attribution_accuracy")
    other_set = name.startswith("smoke") or name.endswith("-devclean")
    print(f"{name + (' *' if other_set else ''):<26}" + "".join(f"{by_m.get(M, float('nan')):>9.4f}" for M in "1234")
          + f"{s.get('spk_acc_n2_cap2', s.get('spk_acc', float('nan'))):>9.4f}{(official if official is not None else float('nan')):>10.4f}")
print("* 검증 세트가 다른 실행 (smoke 계열 또는 dev-clean 전용)\n")

eval_root = Path(DRIVE_ROOT, "eval")
if eval_root.is_dir():
    for set_name in ("eval-t2", "eval-t2-static"):
        print(f"=== [본 평가셋: {set_name} (Spk Acc / cpWER)] ===")
        print(f"{'run/variant':<34}" + "".join(f"{'M=' + M + ' Acc':>10}{'cpWER':>8}" for M in ("4", "3", "2", "1")))
        for score_path in sorted(eval_root.glob(f"*/{set_name}/*/score.json")):
            rep = json.loads(score_path.read_text(encoding="utf-8"))
            label = f"{score_path.parents[2].name}/{score_path.parent.name}"
            by_n = rep.get("breakdown", {}).get("by_n_channels", {})
            cells = []
            for M in ("4", "3", "2", "1"):
                met = (by_n.get(M) or {}).get("metrics", {})
                acc = met.get("speaker_attribution_accuracy", float("nan"))
                cpw = met.get("cpwer", float("nan"))
                cells.append(f"{acc:>10.3f}{cpw:>8.3f}")
            print(f"{label:<34}" + "".join(cells))
        print()

## 7. 전체 통합 요약 블록 — 이 출력을 그대로 복사해 붙여 넣어 주세요

지금까지 Drive에 완료된 **모든 시스템의 검증셋 요약(`runs/`)과 본 평가셋 채점 요약(`eval/`)**을 한 번에 출력합니다.

In [ ]:
from train import SUMMARY_KEYS

all_runs = {}
for path in sorted(Path(DRIVE_ROOT, "runs").glob("*/summary.json")):
    s = json.loads(path.read_text(encoding="utf-8"))
    all_runs[path.parent.name] = {k: s.get(k) for k in SUMMARY_KEYS if s.get(k) is not None}

all_evals = {}
eval_root = Path(DRIVE_ROOT, "eval")
if eval_root.is_dir():
    for run_path in sorted(eval_root.iterdir()):
        if not run_path.is_dir():
            continue
        run_res = {}
        for set_name in ("eval-t2", "eval-t2-static"):
            for score_path in sorted((run_path / set_name).glob("*/score.json")):
                rep = json.loads(score_path.read_text(encoding="utf-8"))
                key = f"{set_name}/{score_path.parent.name}"
                run_res[key] = {
                    "overall": {k: rep["metrics"].get(k) for k in ("speaker_attribution_accuracy", "cpwer", "wer")},
                    "by_M": {m: {k: v["metrics"].get(k) for k in ("speaker_attribution_accuracy", "cpwer", "wer")}
                             for m, v in rep.get("breakdown", {}).get("by_n_channels", {}).items()},
                    "frame_optimal_spk_acc_by_M": rep.get("frame_optimal_spk_acc_by_M"),
                    "eval_t2_run": rep.get("eval_t2_run"),
                }
        if run_res:
            all_evals[run_path.name] = run_res

print("===== VC-SASR ALL RESULTS (paste this back) =====")
print(json.dumps({"validation_runs": all_runs, "eval_t2_runs": all_evals}, ensure_ascii=False))
print("===== END =====")